# Chapter 73: Network Forensics with AI

**Volume 4 — Security Operations**

**A 10 Gbps link generates 1.2 TB of raw packet data per hour.**
Once a packet leaves the wire uncaptured, that evidence is gone forever.
AI does not solve the capture problem — you still need to decide what to capture before
the incident happens. What AI solves is the analysis problem: making sense of
terabytes of evidence fast enough to matter during an active investigation.

### What You Will Build — 5 Forensics Engines

| Demo | Technique | What It Reconstructs |
|------|-----------|----------------------|
| **1. NetFlow Traffic Analyzer** | Session reconstruction + suspicious flow scoring | C2 channels, exfiltration, lateral movement |
| **2. DNS Forensics Engine** | Query timeline + entropy-based tunnel detection | DNS tunneling, DGA activity, beacon domains |
| **3. Protocol Anomaly Detector** | TCP flag analysis + payload pattern inspection | Malformed packets, exploit delivery, covert channels |
| **4. Timeline Reconstructor** | Multi-source correlation by timestamp + bi-temporal | Causal event chains across DNS, flows, auth logs |
| **5. LLM Forensic Narrator** | Abductive reasoning over correlated evidence | Structured incident timeline with MITRE ATT&CK |

### The Core Insight
> **Network forensics is fundamentally abductive: you start from observed evidence
> and work backward to the most plausible cause. The AI does not find the attacker —
> it processes the volume, identifies the pattern, and presents the hypothesis.
> The investigator evaluates it. The chain of custody requires that every step
> is logged, timestamped, and reviewable by a human expert in court.**

*Network analogy: forensic timeline reconstruction is traceroute in reverse —
you have the destination (the breach), and you're reconstructing the hops
the attacker took to get there, using timestamp deltas and flow records
instead of TTL values.*

In [ ]:
# pip install anthropic
import os, json, re, math, time, random
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import defaultdict

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_analyze(prompt: str, max_tokens: int = 200) -> str:
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    return "Simulation: " + prompt[:80] + "..."

print("Setup complete.")

## Demo 1: NetFlow Traffic Analyzer — Session Reconstruction + Suspicious Flow Scoring

**NetFlow/IPFIX** records store connection metadata — who talked to whom, when, and
how much data. They cannot tell you *what* was said (payload), but they tell you
*everything* about communication patterns.

**What we reconstruct from flow records:**
- Session grouping (same 5-tuple, within session window)
- Total bytes transferred per session
- Communication pattern: periodic? burst? single transfer?
- Direction ratio: is this predominantly outbound? (exfiltration indicator)

**Suspicious flow scoring factors:**

| Factor | Score Contribution | What It Indicates |
|--------|--------------------|-------------------|
| Large outbound (>100 MB) | +0.40 | Data exfiltration |
| New external destination | +0.30 | C2 infrastructure contact |
| Off-hours (00:00-06:00) | +0.25 | Attacker operating window |
| Unusual port for service | +0.20 | Protocol tunneling / C2 |
| Very high byte/packet ratio | +0.15 | Large file transfer, compressed data |

*Network analogy: suspicious flow scoring is like SNMP threshold alerting,
but for behavioral patterns rather than utilization counters.
The individual metric doesn't trigger; the combination does.*

In [ ]:
# ── Demo 1: NetFlow Traffic Analyzer ─────────────────────────────────────────
import statistics

@dataclass
class FlowRecord:
    """A single NetFlow/IPFIX flow record."""
    flow_id: str
    src_ip: str
    dst_ip: str
    src_port: int
    dst_port: int
    protocol: str
    bytes_out: int         # bytes from src to dst
    bytes_in: int          # bytes from dst to src (response)
    packets: int
    start_time: float      # Unix timestamp
    duration_sec: float
    flags: str             # TCP flags (SYN, ACK, FIN, RST, PSH)
    dst_is_new: bool       # True if dst_ip never seen in past 30 days


def suspicious_flow_score(flow: FlowRecord) -> Dict:
    """
    Score a flow record for suspicious behavior.
    Returns score (0.0-1.0) and factor breakdown.
    """
    factors = {}
    score = 0.0

    # Factor 1: Large outbound data transfer
    if flow.bytes_out > 100_000_000:   # > 100 MB
        factors["large_outbound"] = 0.40
    elif flow.bytes_out > 10_000_000:  # > 10 MB
        factors["large_outbound"] = 0.20
    else:
        factors["large_outbound"] = 0.0

    # Factor 2: Connection to new/unknown destination
    if flow.dst_is_new:
        factors["new_destination"] = 0.30
    else:
        factors["new_destination"] = 0.0

    # Factor 3: Off-hours activity (midnight to 6am)
    hour = int(time.strftime("%H", time.localtime(flow.start_time)))
    # Use stored hour from the flow's local time context
    flow_hour = getattr(flow, '_hour_override', hour)
    if 0 <= flow_hour <= 6:
        factors["off_hours"] = 0.25
    else:
        factors["off_hours"] = 0.0

    # Factor 4: Unusual port (not standard HTTP/HTTPS/DNS/NTP)
    standard_ports = {80, 443, 53, 123, 22, 25, 143, 993, 587, 465, 110, 995}
    if flow.dst_port not in standard_ports and flow.dst_port < 1024:
        factors["unusual_port"] = 0.20
    elif flow.dst_port > 40000:  # high ephemeral dst port on external host
        factors["unusual_port"] = 0.15
    else:
        factors["unusual_port"] = 0.0

    # Factor 5: High bytes-per-packet (indicates large transfers or compression)
    if flow.packets > 0:
        bpp = flow.bytes_out / flow.packets
        if bpp > 1400:   # near MTU — sustained bulk transfer
            factors["high_bpp"] = 0.15
        else:
            factors["high_bpp"] = 0.0
    else:
        factors["high_bpp"] = 0.0

    # Combined score (sum, capped at 1.0)
    total = min(sum(factors.values()), 1.0)

    return {
        "score": round(total, 3),
        "factors": {k: v for k, v in factors.items() if v > 0},
        "bytes_out_mb": round(flow.bytes_out / 1_000_000, 2),
        "bytes_in_mb":  round(flow.bytes_in  / 1_000_000, 2),
        "ratio": round(flow.bytes_out / max(flow.bytes_in, 1), 2),
    }


# Simulated flow records from a network incident
base_ts = time.time() - 86400  # yesterday

flows = [
    # Normal web traffic
    FlowRecord("f001", "10.0.5.44", "1.1.1.1",       55123, 443, "TCP",
               bytes_out=45_200, bytes_in=2_100_000, packets=1500, start_time=base_ts+32400,
               duration_sec=2.3, flags="SYN,ACK,FIN", dst_is_new=False),
    # First C2 contact — small, new destination, off-hours
    FlowRecord("f002", "10.0.5.44", "185.220.101.47", 59881, 443, "TCP",
               bytes_out=1_240,  bytes_in=890,       packets=8,    start_time=base_ts+11662,
               duration_sec=0.8, flags="SYN,ACK", dst_is_new=True),
    # Lateral movement SMB
    FlowRecord("f003", "10.0.5.44", "10.0.12.5",      61200, 445, "TCP",
               bytes_out=88_420, bytes_in=12_100,    packets=210,  start_time=base_ts+13053,
               duration_sec=4.1, flags="SYN,ACK,PSH", dst_is_new=False),
    # MASSIVE exfiltration — 4.7 GB outbound
    FlowRecord("f004", "10.0.5.44", "185.220.101.47", 59881, 443, "TCP",
               bytes_out=4_983_201_024, bytes_in=4_200, packets=3_412_888,
               start_time=base_ts+19867, duration_sec=7440,
               flags="SYN,ACK,PSH,FIN", dst_is_new=False),
    # Repeated C2 check-in (periodic beacon)
    FlowRecord("f005", "10.0.5.44", "185.220.101.47", 60100, 8080, "TCP",
               bytes_out=320,    bytes_in=280,       packets=4,    start_time=base_ts+11962,
               duration_sec=0.3, flags="SYN,ACK", dst_is_new=False),
    # Normal internal NFS traffic
    FlowRecord("f006", "10.0.5.10", "10.0.1.5",       54321, 2049, "TCP",
               bytes_out=950_000, bytes_in=12_000_000, packets=9000, start_time=base_ts+36000,
               duration_sec=120, flags="SYN,ACK,FIN", dst_is_new=False),
]

# Override hour for off-hours detection (simulate 3am flows)
for f in flows:
    if f.dst_ip == "185.220.101.47":
        f._hour_override = 3
    elif f.dst_ip == "10.0.12.5":
        f._hour_override = 3
    else:
        f._hour_override = 10

print("=" * 72)
print("NETFLOW TRAFFIC ANALYZER — SUSPICIOUS FLOW SCORING")
print("=" * 72)
print(f"{'FlowID':<7} {'Source':<15} {'Destination':<20} {'OutMB':>8} {'In:Out':>7} {'Score':>7}  Factors")
print("-" * 90)

flagged_flows = []
for flow in flows:
    result = suspicious_flow_score(flow)
    dst_display = f"{flow.dst_ip}:{flow.dst_port}"
    factors_short = ",".join(result["factors"].keys()) if result["factors"] else "normal"
    alert = " <-- ALERT" if result["score"] >= 0.40 else ""
    print(f"{flow.flow_id:<7} {flow.src_ip:<15} {dst_display:<20} "
          f"{result['bytes_out_mb']:>8.2f} {result['ratio']:>7.2f} {result['score']:>7.3f}  {factors_short}{alert}")
    if result["score"] >= 0.40:
        flagged_flows.append((flow, result))

print(f"\n[NetFlow] {len(flagged_flows)} flows flagged for investigation")

# LLM triage for most suspicious
if flagged_flows:
    worst = max(flagged_flows, key=lambda x: x[1]["score"])
    flow, result = worst
    print(f"\n[TOP ALERT] Flow {flow.flow_id}: {flow.src_ip} -> {flow.dst_ip}:{flow.dst_port}")
    analysis = llm_analyze(
        f"NetFlow alert: {flow.src_ip} transferred {result['bytes_out_mb']:.1f} MB to "
        f"{flow.dst_ip}:{flow.dst_port} (new destination: {flow.dst_is_new}) at 3am. "
        f"Score: {result['score']:.2f}. Factors: {result['factors']}. "
        f"MITRE ATT&CK stage? Immediate containment action? Under 70 words.",
        max_tokens=100
    )
    print(f"  LLM: {analysis}")

## Demo 2: DNS Forensics Engine — Query Timeline + Tunnel Detection

DNS is one of the most-abused protocols in attacker toolkits because:
1. DNS is almost never blocked outbound (it needs to work for everything)
2. DNS requests look identical to legitimate traffic from the outside
3. DNS can carry arbitrary data in query names and TXT records

**DNS tunneling** encodes exfiltration data directly in DNS query names:
```
Normal DNS:    host.example.com  (short, lowercase, human-readable)
DNS tunnel:    aGVsbG8gd29ybGQ.tunnel.attacker.com  (base64-encoded payload)
```

**Three signals we combine to detect tunneling and beaconing:**

| Signal | Formula | What It Detects |
|--------|---------|----------------|
| **Query entropy** | Shannon entropy of subdomain label | High entropy = encoded data |
| **Query volume** | Queries/minute to same apex domain | Burst = data transfer in progress |
| **Label length** | Length of longest label in FQDN | >40 chars = likely encoded payload |

**Tunnel scoring threshold:**
- Entropy > 3.8 bits/char: suspicious
- Volume > 20 queries/minute to same domain: suspicious  
- Longest label > 35 characters: suspicious
- Any two together = flag for investigation

*Network analogy: DNS query analysis is like inspecting OSPF LSAs for unusually
large or frequent updates — the protocol can carry the data, but the volume
and pattern reveal something has changed from normal operation.*

In [ ]:
# ── Demo 2: DNS Forensics Engine ──────────────────────────────────────────────

@dataclass
class DNSQuery:
    """A single DNS query record from recursive resolver logs."""
    query_id: str
    client_ip: str
    fqdn: str
    query_type: str          # A, AAAA, TXT, MX, CNAME
    timestamp: float
    response_code: str       # NOERROR, NXDOMAIN, SERVFAIL
    response_bytes: int      # size of DNS response
    ttl: int                 # TTL returned in response (very low TTL = fast-flux or C2)


def shannon_entropy_dns(s: str) -> float:
    """Shannon entropy of a string — higher means more random (encoded) content."""
    if not s:
        return 0.0
    freq = defaultdict(int)
    for c in s:
        freq[c] += 1
    total = len(s)
    return -sum((c / total) * math.log2(c / total) for c in freq.values())


def analyze_dns_session(queries: List[DNSQuery]) -> Dict:
    """
    Analyze a set of DNS queries from one client for tunneling indicators.
    Groups queries by apex domain and scores each apex domain.
    """
    # Group by apex domain (last two labels)
    apex_groups = defaultdict(list)
    for q in queries:
        parts = q.fqdn.rstrip('.').split('.')
        if len(parts) >= 2:
            apex = '.'.join(parts[-2:])
        else:
            apex = q.fqdn
        apex_groups[apex].append(q)

    results = []
    for apex, group in apex_groups.items():
        # Entropy of subdomain labels (everything before the apex)
        subdomain_labels = []
        for q in group:
            parts = q.fqdn.rstrip('.').split('.')
            if len(parts) > 2:
                subdomain_labels.extend(parts[:-2])

        avg_entropy = 0.0
        max_label_len = 0
        if subdomain_labels:
            entropies = [shannon_entropy_dns(lbl) for lbl in subdomain_labels if len(lbl) > 3]
            avg_entropy = statistics.mean(entropies) if entropies else 0.0
            max_label_len = max(len(lbl) for lbl in subdomain_labels)

        # Query volume per minute
        if len(group) > 1:
            time_span = max(q.timestamp for q in group) - min(q.timestamp for q in group)
            qpm = (len(group) / max(time_span, 1)) * 60
        else:
            qpm = 0.0

        # Average response TTL (very low = fast-flux)
        avg_ttl = statistics.mean([q.ttl for q in group])

        # Total response bytes (high = data being returned via DNS)
        total_resp_bytes = sum(q.response_bytes for q in group)

        # Tunnel scoring
        tunnel_indicators = 0
        indicator_details = []
        if avg_entropy > 3.8:
            tunnel_indicators += 1
            indicator_details.append(f"high entropy ({avg_entropy:.2f} bits)")
        if qpm > 20:
            tunnel_indicators += 1
            indicator_details.append(f"high volume ({qpm:.1f} qpm)")
        if max_label_len > 35:
            tunnel_indicators += 1
            indicator_details.append(f"long labels ({max_label_len} chars)")
        if avg_ttl < 30:
            tunnel_indicators += 1
            indicator_details.append(f"very low TTL ({avg_ttl:.0f}s)")

        tunnel_score = min(tunnel_indicators * 0.30, 1.0)

        results.append({
            "apex": apex,
            "query_count": len(group),
            "avg_entropy": round(avg_entropy, 3),
            "max_label_len": max_label_len,
            "qpm": round(qpm, 1),
            "avg_ttl": round(avg_ttl, 0),
            "total_resp_kb": round(total_resp_bytes / 1024, 1),
            "tunnel_score": round(tunnel_score, 2),
            "indicators": indicator_details,
            "query_types": list(set(q.query_type for q in group)),
        })

    results.sort(key=lambda x: x["tunnel_score"], reverse=True)
    return results


# Simulated DNS query log from recursive resolver
base_ts = time.time() - 3600

dns_queries = [
    # Normal DNS activity
    DNSQuery("d001", "10.0.5.44", "microsoft.com",         "A",   base_ts+100, "NOERROR",  200, 3600),
    DNSQuery("d002", "10.0.5.44", "google.com",             "A",   base_ts+200, "NOERROR",  180, 300),
    DNSQuery("d003", "10.0.5.44", "windowsupdate.com",      "A",   base_ts+500, "NOERROR",  220, 900),
    # C2 beacon domain — recently registered, low TTL
    DNSQuery("d010", "10.0.5.44", "update-service-cdn.com", "A",   base_ts+835, "NOERROR",  90,  5),
    DNSQuery("d011", "10.0.5.44", "update-service-cdn.com", "A",   base_ts+1135,"NOERROR",  90,  5),
    DNSQuery("d012", "10.0.5.44", "update-service-cdn.com", "A",   base_ts+1435,"NOERROR",  90,  5),
    # DNS tunnel — high-entropy subdomains, TXT queries, rapid fire
    DNSQuery("d020", "10.0.7.22", "aGVsbG8gd29ybGQ.tunnel.exfil-corp.net",  "TXT", base_ts+10,  "NOERROR", 320, 1),
    DNSQuery("d021", "10.0.7.22", "d29ybGQgaGVsbG8.tunnel.exfil-corp.net",  "TXT", base_ts+12,  "NOERROR", 310, 1),
    DNSQuery("d022", "10.0.7.22", "bXkgc2VjcmV0IGRhdGEgaGVyZQ.tunnel.exfil-corp.net", "TXT", base_ts+14, "NOERROR", 890, 1),
    DNSQuery("d023", "10.0.7.22", "dGhpcyBpcyBhIHRlc3QgbWVzc2FnZQ.tunnel.exfil-corp.net", "TXT", base_ts+16, "NOERROR", 760, 1),
    DNSQuery("d024", "10.0.7.22", "c2VjcmV0IGtleXMgZXhmaWx0cmF0aW9u.tunnel.exfil-corp.net", "TXT", base_ts+18, "NOERROR", 920, 1),
    DNSQuery("d025", "10.0.7.22", "bW9yZSBkYXRhIHRvIHNlbmQ.tunnel.exfil-corp.net",    "TXT", base_ts+20, "NOERROR", 440, 1),
    DNSQuery("d026", "10.0.7.22", "ZmluYWwgY2h1bmsgb2YgZXhmaWw.tunnel.exfil-corp.net", "TXT", base_ts+22, "NOERROR", 380, 1),
]

# Analyze all queries from all clients
all_results = analyze_dns_session(dns_queries)

print("=" * 78)
print("DNS FORENSICS ENGINE — TUNNEL DETECTION")
print("=" * 78)
print(f"{'Apex Domain':<35} {'Queries':>7} {'Entropy':>8} {'MaxLen':>7} {'QPM':>6} {'TTL':>6} {'Score':>7}")
print("-" * 78)

for r in all_results:
    alert = " <-- TUNNEL SUSPECTED" if r["tunnel_score"] >= 0.60 else\
            (" <-- BEACON?" if r["tunnel_score"] >= 0.30 else "")
    print(f"{r['apex']:<35} {r['query_count']:>7} {r['avg_entropy']:>8.3f} "
          f"{r['max_label_len']:>7} {r['qpm']:>6.1f} {r['avg_ttl']:>6.0f} "
          f"{r['tunnel_score']:>7.2f}{alert}")
    if r["indicators"]:
        print(f"  Indicators: {', '.join(r['indicators'])}")

# LLM analysis for highest-scored domain
if all_results:
    top = all_results[0]
    print()
    if top["tunnel_score"] >= 0.60:
        analysis = llm_analyze(
            f"DNS forensics alert: domain {top['apex']} shows "
            f"avg entropy {top['avg_entropy']:.2f}, {top['query_count']} queries, "
            f"max label length {top['max_label_len']} chars, "
            f"TTL {top['avg_ttl']:.0f}s. "
            f"Indicators: {top['indicators']}. "
            f"Is this DNS tunneling (T1071.004)? What data may have been exfiltrated? "
            f"Immediate action? Under 80 words.",
            max_tokens=110
        )
        print(f"[LLM] {top['apex']}: {analysis}")

## Demo 3: Protocol Anomaly Detector — TCP Flag Analysis + Payload Patterns

Legitimate TCP traffic follows predictable patterns:
- SYN → SYN-ACK → ACK (three-way handshake)
- Data flows → PSH-ACK → ACK
- FIN-ACK → ACK (graceful close)

Attackers deviate from these patterns in predictable ways:

| Anomaly | TCP Flags | What It Indicates |
|---------|-----------|-------------------|
| **SYN flood** | SYN only, no ACK back | DoS, port scanning |
| **NULL scan** | No flags set | Firewall fingerprinting |
| **XMAS scan** | FIN+PSH+URG | OS fingerprinting |
| **RST flood** | Many RSTs from single src | Connection disruption |
| **SYN with large payload** | SYN + PSH simultaneously | Exploit delivery attempt |

Beyond TCP flags, **payload pattern analysis** detects known exploit signatures:
- High entropy payloads on unusual ports: encrypted C2 or staging tools
- NOP sled patterns in payloads: shellcode delivery
- Specific byte sequences: known malware staging payloads

*Network analogy: protocol anomaly detection is exactly how IOS IPS inspects
traffic against protocol state machines. A packet that violates TCP state
transition rules gets flagged — not because it matches a signature, but because
it breaks the expected state machine.*

In [ ]:
# ── Demo 3: Protocol Anomaly Detector ─────────────────────────────────────────

@dataclass
class PacketRecord:
    """
    Represents a packet or short packet burst from pcap analysis.
    In production: parse from Zeek logs or Wireshark tshark output.
    """
    pkt_id: str
    src_ip: str
    dst_ip: str
    src_port: int
    dst_port: int
    protocol: str
    tcp_flags: List[str]     # e.g. ["SYN"], ["SYN","ACK"], ["FIN","PSH","URG"]
    payload_size: int
    payload_hex: str         # first 32 bytes as hex string (for pattern matching)
    timestamp: float
    window_size: int         # TCP window size (0 = potential DoS)


# Known malicious payload patterns (first bytes)
MALICIOUS_PATTERNS = [
    {"name": "Metasploit NOP sled",   "pattern": "9090909090909090", "severity": "CRITICAL"},
    {"name": "CVE-2017-0144 EternalBlue", "pattern": "ff534d42", "severity": "CRITICAL"},
    {"name": "Log4Shell payload",      "pattern": "7b246a6e64693a", "severity": "CRITICAL"},
    {"name": "Shellcode marker",       "pattern": "4d5a9000",  "severity": "HIGH"},
    {"name": "PowerShell encoded cmd", "pattern": "2d656e63",  "severity": "HIGH"},
]

# Known invalid/suspicious TCP flag combinations
SUSPICIOUS_FLAG_COMBOS = {
    frozenset():                          ("NULL scan",       "HIGH",     "T1046"),
    frozenset(["FIN","PSH","URG"]):       ("XMAS scan",       "HIGH",     "T1046"),
    frozenset(["FIN"]):                   ("FIN scan",        "MEDIUM",   "T1046"),
    frozenset(["SYN","FIN"]):             ("SYN-FIN scan",    "HIGH",     "T1046"),
    frozenset(["SYN","RST"]):             ("SYN-RST invalid", "MEDIUM",   "T1498"),
    frozenset(["SYN","PSH"]):             ("SYN+data",        "HIGH",     "T1190"),  # exploit delivery
}


def payload_entropy(hex_payload: str) -> float:
    """Compute entropy of a hex-encoded payload."""
    if not hex_payload:
        return 0.0
    # Convert hex pairs to byte values
    try:
        bytes_list = [int(hex_payload[i:i+2], 16) for i in range(0, len(hex_payload), 2)]
    except ValueError:
        return 0.0
    if not bytes_list:
        return 0.0
    freq = defaultdict(int)
    for b in bytes_list:
        freq[b] += 1
    n = len(bytes_list)
    return -sum((c/n) * math.log2(c/n) for c in freq.values())


def detect_protocol_anomalies(packet: PacketRecord) -> List[Dict]:
    """Detect protocol anomalies in a single packet record."""
    anomalies = []

    # Check 1: TCP flag combination anomalies
    if packet.protocol == "TCP":
        flag_set = frozenset(packet.tcp_flags)
        if flag_set in SUSPICIOUS_FLAG_COMBOS:
            name, severity, mitre = SUSPICIOUS_FLAG_COMBOS[flag_set]
            anomalies.append({
                "type": "FLAG_ANOMALY",
                "name": name,
                "severity": severity,
                "mitre": mitre,
                "detail": f"Flags: {sorted(packet.tcp_flags)}",
            })

    # Check 2: Payload pattern matching
    if packet.payload_hex:
        payload_lower = packet.payload_hex.lower().replace(' ', '')
        for pattern in MALICIOUS_PATTERNS:
            if pattern["pattern"] in payload_lower:
                anomalies.append({
                    "type": "PAYLOAD_MATCH",
                    "name": pattern["name"],
                    "severity": pattern["severity"],
                    "mitre": "T1059",
                    "detail": f"Pattern matched at offset in {len(packet.payload_hex)//2}-byte payload",
                })

    # Check 3: High-entropy payload on non-HTTPS port (covert channel indicator)
    if packet.payload_size > 100 and packet.dst_port not in {443, 8443, 22}:
        entropy = payload_entropy(packet.payload_hex)
        if entropy > 7.5:  # near-maximum entropy = encrypted/compressed data
            anomalies.append({
                "type": "HIGH_ENTROPY_PAYLOAD",
                "name": "Encrypted/compressed payload on non-TLS port",
                "severity": "MEDIUM",
                "mitre": "T1573",
                "detail": f"Entropy={entropy:.2f} on port {packet.dst_port} ({packet.payload_size} bytes)",
            })

    # Check 4: Zero window size (potential DoS)
    if packet.window_size == 0 and "SYN" not in packet.tcp_flags:
        anomalies.append({
            "type": "ZERO_WINDOW",
            "name": "TCP zero window",
            "severity": "LOW",
            "mitre": "T1498",
            "detail": "Potential DoS or connection stall",
        })

    return anomalies


# Simulated packet captures
test_packets = [
    PacketRecord("p001", "10.0.5.44", "10.0.12.5",    54321, 445, "TCP",
                 ["SYN","ACK"], 0, "", time.time(), 65535),
    PacketRecord("p002", "203.0.113.5", "10.0.1.10",  12345, 80,  "TCP",
                 ["SYN","PSH"], 320, "7b246a6e64693a6c6461703a2f2f6174742e636f6d2f657869",
                 time.time(), 1024),
    PacketRecord("p003", "198.51.100.7", "10.0.4.20", 51234, 23,  "TCP",
                 [], 0, "", time.time(), 0),   # NULL scan
    PacketRecord("p004", "198.51.100.8", "10.0.5.44", 9999,  445, "TCP",
                 ["FIN","PSH","URG"], 0, "", time.time(), 0),  # XMAS scan
    PacketRecord("p005", "10.0.3.88",   "10.0.99.1",  44444, 8888,"TCP",
                 ["PSH","ACK"], 512,
                 "f3c8a2e91b047d56c8a2e91b047d56c8a2e91b047d56c8a2e91b047d5600",
                 time.time(), 8192),  # high entropy on unusual port
    PacketRecord("p006", "10.0.2.55",   "10.0.12.5",  43210, 445, "TCP",
                 ["PSH","ACK"], 128, "ff534d423500000000000000000000000000fffe00000000",
                 time.time(), 8192),  # EternalBlue pattern
    PacketRecord("p007", "10.0.1.100",  "8.8.8.8",    51000, 53,  "UDP",
                 [], 42, "0001000100000000", time.time(), 0),  # normal DNS
]

print("=" * 72)
print("PROTOCOL ANOMALY DETECTOR — PACKET ANALYSIS")
print("=" * 72)

all_anomalies = []
for pkt in test_packets:
    anomalies = detect_protocol_anomalies(pkt)
    src = f"{pkt.src_ip}:{pkt.src_port}"
    dst = f"{pkt.dst_ip}:{pkt.dst_port}"
    flags_str = '+'.join(pkt.tcp_flags) if pkt.tcp_flags else 'NULL'
    if anomalies:
        print(f"\n[ALERT] {pkt.pkt_id}: {src} -> {dst} [{flags_str}]")
        for a in anomalies:
            print(f"  [{a['severity']}] {a['name']} ({a['mitre']}) — {a['detail']}")
            all_anomalies.append((pkt, a))
    else:
        print(f"  [OK]   {pkt.pkt_id}: {src} -> {dst} [{flags_str}] — normal")

print(f"\n[Anomaly Detector] {len(all_anomalies)} anomalies across {len(test_packets)} packets")

# LLM for most severe
critical = [(p, a) for p, a in all_anomalies if a["severity"] == "CRITICAL"]
if critical:
    pkt, anomaly = critical[0]
    analysis = llm_analyze(
        f"Packet anomaly: {pkt.src_ip} -> {pkt.dst_ip}:{pkt.dst_port}. "
        f"Anomaly: {anomaly['name']} ({anomaly['mitre']}). Detail: {anomaly['detail']}. "
        f"What attack technique is this? Immediate containment? Under 70 words.",
        max_tokens=100
    )
    print(f"\n[LLM] {anomaly['name']}: {analysis}")

## Demo 4: Timeline Reconstructor — Multi-Source Correlation + Bi-Temporal Modeling

The hardest part of forensics is not finding individual events — it's determining
which events are causally related and reconstructing the sequence.

**Bi-temporal evidence modeling stores two timestamps for every event:**
- **Event Time (T)**: when the event actually happened (the packet timestamp,
  the auth log time, the DNS query time)
- **Information Time (T')**: when your collection system ingested this evidence
  (SIEM ingestion time, collector export time)

**Why this matters:**
```
Legitimate event:  T=03:14:22, T'=03:14:25  (3-second collection lag — normal)
Suspicious gap:    T=03:14:22, T'=11:45:33  (8-hour gap — why so late?)
Impossible event:  T=03:14:22, T'=03:12:00  (T' before T — timestamp manipulation!)
```

**Correlation approach:**
1. Merge all events (DNS, NetFlow, auth, packets) into a unified timeline
2. Sort by event time (T)
3. Score causal linkages: events within 120-second window on same host get
   higher causal confidence
4. Flag bi-temporal anomalies (large T-T' gaps, impossible timestamps)
5. Identify kill-chain stage (initial access → discovery → lateral → exfil)

*Network analogy: timeline reconstruction is BGP table convergence analysis.
You're correlating route update timestamps across multiple BGP speakers to
determine which router learned what, and in what order.*

In [ ]:
# ── Demo 4: Timeline Reconstructor ────────────────────────────────────────────
from datetime import datetime, timezone

@dataclass
class EvidenceEvent:
    """
    A single forensic evidence item with bi-temporal timestamps.
    Supports events from any source: DNS, NetFlow, auth logs, packet capture.
    """
    event_id: str
    source_type: str       # "DNS", "NETFLOW", "AUTH", "PCAP", "SYSLOG"
    host: str              # affected host or source of event
    event_time: float      # T: when it actually happened (Unix timestamp)
    info_time: float       # T': when the SIEM/collector ingested this event
    description: str
    ioc: str               # indicator of compromise if any
    kill_chain_stage: str  # MITRE ATT&CK kill chain stage
    confidence: float      # 0.0-1.0


def format_ts(ts: float) -> str:
    """Format a Unix timestamp as a readable UTC string."""
    return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


def check_bi_temporal_anomaly(event: EvidenceEvent) -> Optional[str]:
    """
    Check for bi-temporal anomalies:
    - T' before T: physically impossible (timestamp manipulation)
    - T' > T + 4 hours: suspicious collection delay
    - T on exactly round boundary: potential manual timestamp setting
    """
    lag = event.info_time - event.event_time
    if lag < 0:
        return f"IMPOSSIBLE: event_time > info_time by {abs(lag):.0f}s — timestamp manipulation?"
    if lag > 14400:  # 4 hours
        hours = lag / 3600
        return f"SUSPICIOUS: {hours:.1f}h collection lag — why did this event arrive so late?"
    # Check if event_time is suspiciously round (on-the-hour or exact minute)
    if event.event_time % 3600 == 0:
        return "SUSPICIOUS: event_time is exactly on-the-hour — potential manual timestamp"
    return None


def compute_causal_links(events: List[EvidenceEvent], window_sec: int = 120) -> List[tuple]:
    """
    Find pairs of events that may be causally linked:
    - Same or related host
    - Within time window
    - Progressive kill chain stage
    Returns: list of (event_a, event_b, confidence) tuples
    """
    STAGE_ORDER = {
        "INITIAL_ACCESS": 1, "EXECUTION": 2, "DISCOVERY": 3,
        "LATERAL_MOVEMENT": 4, "COLLECTION": 5, "EXFILTRATION": 6
    }
    links = []
    sorted_events = sorted(events, key=lambda e: e.event_time)

    for i, a in enumerate(sorted_events):
        for b in sorted_events[i+1:]:
            time_gap = b.event_time - a.event_time
            if time_gap > window_sec:
                break  # events are sorted, no further matches possible

            # Same host or related host (lateral movement: different hosts OK)
            same_host = (a.host == b.host)
            cross_host = (a.kill_chain_stage == "LATERAL_MOVEMENT")

            if not (same_host or cross_host):
                continue

            # Progressive kill chain stage increases confidence
            stage_a = STAGE_ORDER.get(a.kill_chain_stage, 0)
            stage_b = STAGE_ORDER.get(b.kill_chain_stage, 0)
            progressive = stage_b >= stage_a

            # Compute link confidence
            confidence = 0.5
            if same_host:       confidence += 0.2
            if progressive:     confidence += 0.2
            if time_gap < 60:   confidence += 0.1  # very close in time

            links.append((a, b, round(min(confidence, 1.0), 2)))

    return links


# Simulated multi-source evidence for the incident
T0 = 1710468862.0  # 2024-03-15 03:14:22 UTC

evidence = [
    EvidenceEvent("e01", "DNS",     "10.0.5.44", T0 - 27,   T0 - 24,   # 03:13:55
                  "DNS query: update-service-cdn.com (domain registered 3 days ago)",
                  "update-service-cdn.com", "INITIAL_ACCESS", 0.75),
    EvidenceEvent("e02", "NETFLOW", "10.0.5.44", T0,        T0 + 3,    # 03:14:22
                  "First TCP connection to 185.220.101.47:443 — 1,240 bytes (handshake)",
                  "185.220.101.47", "INITIAL_ACCESS", 0.80),
    EvidenceEvent("e03", "NETFLOW", "10.0.5.44", T0 + 1991, T0 + 1994, # 03:47:33
                  "SMB connection 10.0.5.44 -> 10.0.12.5:445 — 88,420 bytes",
                  "10.0.12.5", "LATERAL_MOVEMENT", 0.70),
    EvidenceEvent("e04", "DNS",     "10.0.5.44", T0 + 2039, T0 + 2042, # 03:48:01
                  "DNS query: fileserver.corp.internal (internal DNS)",
                  "fileserver.corp.internal", "DISCOVERY", 0.65),
    EvidenceEvent("e05", "AUTH",    "10.0.5.44", T0 + 2090, T0 + 2093, # 03:49:12
                  "svc-backup LogonSuccess on 10.0.5.44 — first-ever logon for this account",
                  "svc-backup", "LATERAL_MOVEMENT", 0.90),
    EvidenceEvent("e06", "AUTH",    "10.0.12.5", T0 + 2843, T0 + 2850, # 04:02:05 — 7s lag
                  "svc-backup FileAccess \\\\Finance\\\\Q4-Reports\\ — 847 files in 4 minutes",
                  "\\\\Finance\\\\Q4-Reports", "COLLECTION", 0.95),
    EvidenceEvent("e07", "NETFLOW", "10.0.5.44", T0 + 7605, T0 + 7608, # 05:31:07
                  "4.98 GB outbound transfer 10.0.5.44 -> 185.220.101.47:443",
                  "185.220.101.47", "EXFILTRATION", 0.98),
    # Suspicious: this event arrives in SIEM 6 hours late
    EvidenceEvent("e08", "SYSLOG", "10.0.5.44",  T0 - 300,  T0 + 21300, # 8.25h lag!
                  "Antivirus disabled on 10.0.5.44 (svchost stopped)",
                  "antivirus_disabled", "EXECUTION", 0.85),
]

print("=" * 78)
print("TIMELINE RECONSTRUCTOR — MULTI-SOURCE CORRELATION + BI-TEMPORAL ANALYSIS")
print("=" * 78)

sorted_evidence = sorted(evidence, key=lambda e: e.event_time)
print(f"{'ID':<5} {'T (Event Time)':<24} {'Stage':<20} {'Src':<8} {'Conf':>5}  Description")
print("-" * 100)

bitemporal_flags = []
for e in sorted_evidence:
    ts_str = format_ts(e.event_time)
    anomaly = check_bi_temporal_anomaly(e)
    flag = " [!]" if anomaly else ""
    print(f"{e.event_id:<5} {ts_str:<24} {e.kill_chain_stage:<20} {e.source_type:<8} "
          f"{e.confidence:>5.2f}  {e.description[:60]}{flag}")
    if anomaly:
        print(f"       BI-TEMPORAL ANOMALY: {anomaly}")
        bitemporal_flags.append((e, anomaly))

# Causal link analysis
print(f"\n[Causal Links] Analyzing pairs within 120-second windows...")
links = compute_causal_links(evidence)
print(f"Found {len(links)} potential causal links")

for a, b, conf in sorted(links, key=lambda x: x[2], reverse=True)[:4]:
    gap = b.event_time - a.event_time
    print(f"  {a.event_id} -> {b.event_id} | gap={gap:.0f}s | conf={conf:.2f} | "
          f"{a.kill_chain_stage} -> {b.kill_chain_stage}")

print(f"\n[Summary] Kill chain coverage:")
stages_seen = sorted(set(e.kill_chain_stage for e in evidence),
                     key=lambda s: ["INITIAL_ACCESS","EXECUTION","DISCOVERY",
                                    "LATERAL_MOVEMENT","COLLECTION","EXFILTRATION"].index(s)
                     if s in ["INITIAL_ACCESS","EXECUTION","DISCOVERY",
                              "LATERAL_MOVEMENT","COLLECTION","EXFILTRATION"] else 99)
print("  " + " -> ".join(stages_seen))
print(f"  Bi-temporal anomalies: {len(bitemporal_flags)} events require investigation")

## Demo 5: LLM Forensic Narrative Generator — Abductive Reasoning over Evidence

The four previous demos generate structured evidence. This demo uses an LLM to
reason over that evidence **abductively** — forming the most plausible hypothesis
that explains all observed data simultaneously.

**Abductive reasoning in forensics:**
- Deductive: "All ransomware encrypts files. This encrypted files. This is ransomware."
  *(fails for novel techniques that use legitimate tools)*
- Abductive: "I see DNS to a new domain, followed by a small connection, then lateral
  movement, then credential use, then bulk file access, then 4.7 GB outbound.
  The most plausible explanation is..."

The LLM analyst receives a structured evidence package and outputs:
1. **INCIDENT TIMELINE**: chronological reconstruction
2. **HYPOTHESIS**: most plausible attack chain with MITRE ATT&CK techniques
3. **KEY EVIDENCE**: top 3 most significant indicators
4. **GAPS**: what evidence is missing and why it matters
5. **NEXT STEPS**: specific investigative actions for the analyst

**Chain-of-custody note:**
> Every LLM inference is logged with: evidence IDs used, model version,
> timestamp (T' for when inference ran), and the analyst who reviewed the output.
> AI forensic output without an audit trail cannot be used in legal proceedings.
> The immutable log is not optional.

In [ ]:
# ── Demo 5: LLM Forensic Narrative Generator ──────────────────────────────────

@dataclass
class ForensicAnalysisJob:
    """
    A complete forensic analysis job — evidence in, narrative out.
    Maintains an audit log of all LLM inferences for chain-of-custody.
    """
    job_id: str
    analyst: str
    evidence: List[EvidenceEvent]
    flow_findings: List[Dict]       # from Demo 1
    dns_findings: List[Dict]        # from Demo 2
    anomaly_findings: List[Dict]    # from Demo 3
    # Populated after analysis
    audit_log: List[Dict] = field(default_factory=list)
    narrative: str = ""

    def _log_inference(self, evidence_ids: List[str], model: str,
                       prompt_hash: str, output_summary: str):
        """Append an immutable audit record for this LLM inference."""
        self.audit_log.append({
            "job_id": self.job_id,
            "analyst": self.analyst,
            "info_time": time.time(),             # T' — when inference ran
            "model": model,
            "evidence_ids": evidence_ids,
            "prompt_hash": prompt_hash,
            "output_preview": output_summary[:100],
            "status": "LLM_INFERENCE_COMPLETE",
        })

    def build_evidence_package(self) -> str:
        """Build a structured evidence text package for the LLM."""
        lines = []

        lines.append("=== CORRELATED INCIDENT TIMELINE ===")
        for e in sorted(self.evidence, key=lambda x: x.event_time):
            ts = format_ts(e.event_time)
            anomaly = check_bi_temporal_anomaly(e)
            anom_note = f" [BI-TEMPORAL: {anomaly}]" if anomaly else ""
            lines.append(f"[{ts}] [{e.source_type}] [{e.kill_chain_stage}] "
                         f"{e.host}: {e.description} (conf={e.confidence:.0%}){anom_note}")

        if self.flow_findings:
            lines.append("\n=== NETFLOW SUSPICIOUS FLOWS ===")
            for flow, result in self.flow_findings:
                lines.append(f"  {flow.src_ip} -> {flow.dst_ip}:{flow.dst_port} | "
                             f"{result['bytes_out_mb']:.1f} MB out | "
                             f"score={result['score']:.2f} | factors={list(result['factors'].keys())}")

        if self.dns_findings:
            flagged_dns = [r for r in self.dns_findings if r["tunnel_score"] >= 0.30]
            if flagged_dns:
                lines.append("\n=== DNS SUSPICIOUS DOMAINS ===")
                for r in flagged_dns:
                    lines.append(f"  {r['apex']}: score={r['tunnel_score']:.2f}, "
                                 f"indicators={r['indicators']}")

        if self.anomaly_findings:
            lines.append("\n=== PROTOCOL ANOMALIES ===")
            for pkt, anomaly in self.anomaly_findings:
                lines.append(f"  [{anomaly['severity']}] {pkt.src_ip}->{pkt.dst_ip}:{pkt.dst_port}: "
                             f"{anomaly['name']} ({anomaly['mitre']})")

        return "\n".join(lines)

    def generate_narrative(self) -> str:
        """Call LLM to generate forensic narrative using abductive reasoning."""
        evidence_package = self.build_evidence_package()
        evidence_ids = [e.event_id for e in self.evidence]

        # Hash the prompt for audit log integrity
        import hashlib
        prompt_hash = hashlib.sha256(evidence_package.encode()).hexdigest()[:16]

        system_prompt = (
            "You are a senior network forensics investigator. "
            "Analyze multi-source evidence using abductive reasoning: identify the most plausible "
            "hypothesis that explains ALL observed evidence simultaneously. "
            "Use MITRE ATT&CK technique IDs. Be specific and direct.\n"
            "Structure your response EXACTLY as:\n"
            "INCIDENT TIMELINE: [4-6 bullet chronological reconstruction]\n"
            "HYPOTHESIS: [one paragraph — most plausible attack chain]\n"
            "KEY EVIDENCE: [top 3 most significant indicators, one line each]\n"
            "GAPS: [what evidence is missing and why it matters]\n"
            "NEXT STEPS: [3 specific investigative actions with tool/query specifics]"
        )

        user_prompt = (
            f"Reconstruct the incident. Primary affected host: 10.0.5.44. All times UTC.\n\n"
            f"{evidence_package}"
        )

        if USE_API:
            from anthropic import Anthropic
            narr_client = Anthropic()
            resp = narr_client.messages.create(
                model="claude-sonnet-4-5-20250514",
                max_tokens=600,
                system=system_prompt,
                messages=[{"role": "user", "content": user_prompt}]
            )
            narrative = resp.content[0].text
        else:
            narrative = (
                "[SIMULATION MODE — set ANTHROPIC_API_KEY for real forensic narrative]\n\n"
                "INCIDENT TIMELINE:\n"
                "• 03:13:55 UTC — 10.0.5.44 queried update-service-cdn.com (3-day-old domain, INITIAL_ACCESS indicator)\n"
                "• 03:14:22 UTC — First C2 connection to 185.220.101.47:443 (1,240 bytes — handshake/implant callback)\n"
                "• ~03:09 UTC — Antivirus disabled on 10.0.5.44 (arrived SIEM 8h late — bi-temporal anomaly)\n"
                "• 03:47-49 UTC — Internal reconnaissance: SMB to 10.0.12.5, DNS for fileserver, svc-backup account used\n"
                "• 04:02 UTC — Bulk file access: 847 Finance/Q4-Reports files in 4 minutes (COLLECTION stage)\n"
                "• 05:31 UTC — 4.98 GB exfiltrated to 185.220.101.47:443 over 2-hour window\n\n"
                "HYPOTHESIS: 10.0.5.44 was compromised via a malware dropper that performed C2 callback to\n"
                "185.220.101.47 (T1071.001). The attacker pre-disabled AV (T1562.001) before the C2 beacon.\n"
                "Using svc-backup credentials harvested in memory (T1003.001), lateral movement reached 10.0.12.5\n"
                "where Finance data was staged and exfiltrated (T1048) via the C2 HTTPS channel.\n\n"
                "KEY EVIDENCE:\n"
                "1. 4.98 GB outbound to C2 IP (e07) — confirms exfiltration; ratio 1,186,476:1 out:in\n"
                "2. svc-backup first-ever logon from 10.0.5.44 (e05) — credential theft confirmed\n"
                "3. Antivirus disabled 5 min before C2 contact (e08) — indicates pre-compromise foothold\n\n"
                "GAPS: No FPCap payload data — cannot confirm what exploit delivered the initial implant.\n"
                "No memory forensics — cannot identify exact malware family or confirm Mimikatz usage.\n\n"
                "NEXT STEPS:\n"
                "1. Pull FPCap for 10.0.5.44 inbound traffic 02:00-03:15 UTC — look for exploit payload delivery\n"
                "2. Memory forensics on 10.0.5.44 (Volatility3 pslist+malfind) — identify malware process and injected code\n"
                "3. Query SIEM: all logins using svc-backup account across all hosts, past 7 days — determine lateral extent"
            )

        self._log_inference(evidence_ids, "claude-sonnet-4-5-20250514",
                            prompt_hash, narrative[:100])
        self.narrative = narrative
        return narrative


# ── Run the forensic narrative generator ──────────────────────────────────────
print("=" * 70)
print("LLM FORENSIC NARRATIVE GENERATOR")
print("=" * 70)

# Gather flagged flows and anomalies from earlier demos
flagged_flows_for_narrative = [
    (f, suspicious_flow_score(f)) for f in flows if suspicious_flow_score(f)["score"] >= 0.40
]
all_dns_results = analyze_dns_session(dns_queries)
packet_anomalies = []
for pkt in test_packets:
    for a in detect_protocol_anomalies(pkt):
        if a["severity"] in ("CRITICAL", "HIGH"):
            packet_anomalies.append((pkt, a))

job = ForensicAnalysisJob(
    job_id="FORENSIC-2024-03-15-001",
    analyst="ed@vexpertai.com",
    evidence=evidence,
    flow_findings=flagged_flows_for_narrative,
    dns_findings=all_dns_results,
    anomaly_findings=packet_anomalies,
)

print("Building evidence package from all 5 detection engines...")
pkg = job.build_evidence_package()
print(f"Evidence package: {len(pkg)} chars across {len(evidence)} events")
print(f"Flagged flows: {len(flagged_flows_for_narrative)} | "
      f"DNS findings: {len([r for r in all_dns_results if r['tunnel_score']>=0.30])} | "
      f"Protocol anomalies: {len(packet_anomalies)}")
print()
print("Generating forensic narrative (LLM abductive reasoning)...")
narrative = job.generate_narrative()

print("=" * 70)
print("FORENSIC INCIDENT NARRATIVE")
print("=" * 70)
print(narrative)

print()
print("=" * 70)
print("CHAIN OF CUSTODY AUDIT LOG")
print("=" * 70)
for entry in job.audit_log:
    print(f"  Job: {entry['job_id']}")
    print(f"  Analyst: {entry['analyst']}")
    print(f"  Info-Time (T'): {format_ts(entry['info_time'])}")
    print(f"  Model: {entry['model']}")
    print(f"  Evidence IDs: {entry['evidence_ids']}")
    print(f"  Prompt hash (SHA-256 prefix): {entry['prompt_hash']}")
    print(f"  Status: {entry['status']}")
    print()
print("IMPORTANT: This audit log entry must be preserved with the evidence.")
print("The prompt hash proves the exact evidence set used for this inference.")
print("Without this log, AI-generated narrative cannot be used in legal proceedings.")

## Summary: What You Built

You now have a working **AI-Powered Network Forensics** system with 5 analysis engines:

| Engine | Key Technique | Evidence It Produces |
|--------|--------------|---------------------|
| **NetFlow Analyzer** | Suspicious flow scoring (5 factors) | C2 channels, exfiltration volume, lateral movement |
| **DNS Forensics** | Entropy + volume + label length | DNS tunneling, beacon domains, fast-flux C2 |
| **Protocol Anomaly** | TCP flag state machine + payload patterns | Scanners, exploit delivery, covert channels |
| **Timeline Reconstructor** | Bi-temporal correlation + causal links | Full kill-chain sequence with confidence scores |
| **Forensic Narrator** | LLM abductive reasoning | MITRE-mapped narrative + investigative next steps |

### The Bi-Temporal Rule
```
Every forensic event stores TWO timestamps:
  T  = event time  (when it happened in the real world)
  T' = info time   (when your SIEM/collector ingested it)

T' < T  = impossible = timestamp manipulation attack
T' - T > 4h = suspicious collection delay, investigate
T on exact hour boundary = potential manual timestamp setting
```

### The Chain-of-Custody Contract
> **Every LLM inference in a forensic investigation must be logged with:**
> model version, evidence IDs used, prompt hash, information timestamp (T'),
> and the analyst who reviewed and approved the output.
> AI forensic output without this audit trail cannot be used in legal proceedings.
> The UK Post Office Horizon scandal is the warning: unauditable computer-generated
> evidence caused hundreds of wrongful convictions.

### Non-Negotiable Human-in-the-Loop Guardrail
> **The forensic narrative is a hypothesis, not a verdict.**
> AI applies abductive reasoning at scale. A qualified forensic investigator
> reviews every hypothesis, evaluates it against the full evidence picture,
> and takes professional responsibility for any conclusion submitted to legal,
> compliance, or executive audiences. The AI does not replace the investigator.
> It reduces analysis time from days to hours so the investigator can focus
> on the judgment calls that require human expertise.

### Production Upgrade Path
```
Simulated flow records     ->  Zeek/Arkime API or IPFIX collector query
Simulated DNS queries      ->  Recursive resolver logs (Bind9, Unbound, Windows DNS)
Simulated packet records   ->  Wireshark tshark JSON export or Zeek conn.log
In-memory evidence list    ->  Elasticsearch or OpenSearch SIEM query API
Single LLM call            ->  Multi-agent: NetFlow agent + DNS agent + PCAP agent + Narrator
Local audit log            ->  Append-only immutable store (S3 object lock, Worm storage)
```

**Next: Chapter 75 — Network Anomaly Detection** — building the continuous models
that generate the initial alerts that trigger forensic investigations like this one.

In [ ]:
# ── Production Deployment Checklist ───────────────────────────────────────────
print("CHAPTER 73: PRODUCTION DEPLOYMENT CHECKLIST")
print("=" * 64)

checklist = [
    # Capture infrastructure
    ("Capture Infrastructure",  "FPCap at internet border + CDE boundary (72h retention)"),
    ("Capture Infrastructure",  "NetFlow/IPFIX everywhere else (90-day retention)"),
    ("Capture Infrastructure",  "Recursive DNS query logging — ALL resolvers"),
    ("Capture Infrastructure",  "SHA-256 hash all capture files at time of collection"),
    # Clock discipline
    ("Clock Discipline",        "Authenticated NTP (MD5 or NTPsec) on ALL evidence sources"),
    ("Clock Discipline",        "Alert on clock drift > 1 second from reference"),
    ("Clock Discipline",        "Store both T (event) and T' (ingest) for every record"),
    ("Clock Discipline",        "Flag T' < T or T' - T > 4h as bi-temporal anomaly"),
    # Analysis tuning
    ("Analysis Tuning",         "NetFlow score threshold: 0.40+ = flag, 0.70+ = immediate"),
    ("Analysis Tuning",         "DNS tunnel: entropy > 3.8 OR label > 35 chars = investigate"),
    ("Analysis Tuning",         "TCP NULL/XMAS scan: always alert regardless of score"),
    ("Analysis Tuning",         "Causal link window: 120s for same-host, 600s for lateral"),
    # LLM integration
    ("LLM Integration",         "Model: claude-haiku-4-5-20251001 (triage), claude-sonnet-4-5-20250514 (narrative)"),
    ("LLM Integration",         "Always structure output: Timeline/Hypothesis/Evidence/Gaps/NextSteps"),
    ("LLM Integration",         "Include MITRE ATT&CK technique IDs in every hypothesis"),
    ("LLM Integration",         "Narrative confidence: distinguish evidence-based vs inferred"),
    # Chain of custody
    ("Chain of Custody",        "MANDATORY: immutable audit log for every LLM inference"),
    ("Chain of Custody",        "Log: job_id, analyst, T', model, evidence_ids, prompt_hash"),
    ("Chain of Custody",        "Human expert must review ALL narrative before submission"),
    ("Chain of Custody",        "Crypto-shredding for GDPR: encrypt PII, destroy key on request"),
]

current_cat = ""
for category, item in checklist:
    if category != current_cat:
        print(f"\n[{category}]")
        current_cat = category
    print(f"  ✓ {item}")

print()
print("=" * 64)
print("KEY FORMULAS")
print("=" * 64)
print("Flow score     = Σ(large_outbound + new_dst + off_hours + unusual_port + high_bpp)")
print("DNS tunnel     = tunnel_indicators × 0.30  (high_entropy + high_qpm + long_label + low_TTL)")
print("Causal conf    = 0.5 + same_host(+0.2) + progressive_stage(+0.2) + <60s_gap(+0.1)")
print("Bi-temporal    = info_time - event_time  (<0 = impossible | >4h = suspicious)")
print()
print("EVIDENCE RETENTION RECOMMENDATIONS:")
print("  FPCap (full packet): 72 hours minimum | 7 days for crown jewel segments")
print("  NetFlow/IPFIX:       90 days minimum  | 1 year for compliance-regulated segments")
print("  DNS query logs:      90 days minimum  | required for tunnel detection over time")
print("  Auth/RADIUS logs:    1 year minimum   | legal hold: indefinite")
print("  LLM audit log:       match legal hold | immutable storage mandatory")
print()
print("Chapter 73 complete. Capture everything. Analyze fast. Log every inference.")